### Task 2: AdaBoost (Adaptive Boosting)
#### **Objective**
In this task, you will implement the **AdaBoost** algorithm for binary classification. AdaBoost is an ensemble method that combines multiple "weak learners" (typically simple decision trees) to create a strong classifier.

#### **Overview**
The AdaBoost algorithm works by iteratively training weak learners on the dataset. In each iteration, it adjusts the weights of the training samples so that the next learner focuses more on the samples that were misclassified by the previous ones.

The algorithm proceeds as follows (for $T$ rounds):
1. **Initialize weights**: Assign equal weights $w_i = 1/n$ to all $n$ training samples.
2. **Iterate $t=1$ to $T$**:
   - **(a) Fit a weak learner**: Train a base classifier $h_t$ using the weighted samples.
   - **(b) Compute weighted error**: Calculate the error $\varepsilon_t$ as the sum of weights of misclassified samples.
   - **(c) Compute learner weight**: Calculate $\alpha_t = \frac{1}{2} \ln\left(\frac{1-\varepsilon_t}{\varepsilon_t}\right)$.
   - **(d) Update sample weights**: Increase weights for misclassified samples: $w_i \leftarrow w_i \cdot \exp(-\alpha_t y_i h_t(x_i))$.
   - **(e) Normalize weights**: Scale $w_i$ so that $\sum w_i = 1$.
3. **Final Prediction**: The final model aggregates predictions: $F_T(x) = \sum_{t=1}^T \alpha_t h_t(x)$, and the predicted class is $\text{sign}(F_T(x))$.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

class AdaBoost:
    """
    AdaBoost (Adaptive Boosting) classifier for binary classification.
    """
    
    def __init__(self, n_estimators=50, seed=None):
        self.n_estimators = n_estimators
        self.seed = seed
        self.alphas = []
        self.models = []
        
    def _compute_error(self, y, y_pred, w):
        """
        Calculates the weighted training error.
        """
            
        error = sum(w * (y_pred != y))
        return error
    
    def _compute_alpha(self, error):
        """
        Computes the weight (alpha) of the current weak learner.
        """
        epsilon = 1e-10  # Tiny constant to avoid division by zero
        alpha = (1/2)*np.log((1-error+epsilon)/(error+epsilon))
        return alpha
    
    def _update_weights(self, w, alpha, y, y_pred):
        """
        Updates and normalizes sample weights.
        """
        for i in range(len(w)):
            w[i] = w[i] * np.e ** (-alpha * y[i] * y_pred[i])
 
        w = w/sum(w)
        return w
        
    def fit(self, X, y):
        """
        Trains the AdaBoost model.
        """
        n_samples, n_features = X.shape
        
        self.models = []
        self.alphas = []
        
        if set(np.unique(y)) == {0, 1}:
            y = np.where(y == 0, -1, 1)
            
        w = np.full(n_samples, 1.0 / n_samples)
        
        if self.seed is not None:
            np.random.seed(self.seed)
            
        for t in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1, random_state=self.seed)
            stump.fit(X, y, sample_weight=w)
            
            y_pred = stump.predict(X)
            
            error = self._compute_error(y, y_pred, w)
            
            alpha = self._compute_alpha(error)
            
            w = self._update_weights(w, alpha, y, y_pred)
            self.models.append(stump)
            self.alphas.append(alpha)
            
    def predict(self, X):
        """
        Predicts class labels for X.
        """
        weak_preds = np.array([stump.predict(X) for stump in self.models])
        weighted_sum = np.dot(self.alphas, weak_preds)
        final_preds = np.sign(weighted_sum)
        final_preds[final_preds == 0] = 1
        
        return final_preds



11.512925465020228
0.25
[3.33332222e-01 3.33332222e-01 3.33332222e-01 3.33332222e-06]


### Verification
Let's test your implementation on the Breast Cancer dataset.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X = data.data
y = data.target

y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = AdaBoost(n_estimators=50, seed=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Test Accuracy: 0.8947
